In [ ]:
import scipy.io
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


In [ ]:

data = scipy.io.loadmat('../Xtrain.mat')
X = data['Xtrain'].flatten().astype(np.float32)  # shape: (1000,)

print(f"Length: {len(X)}")
print(f"Min: {X.min()}, Max: {X.max()}, Mean: {X.mean():.2f}, Std: {X.std():.2f}")

plt.figure(figsize=(12, 4))
plt.plot(X)
plt.title("Raw Laser Measurement Data")
plt.xlabel("Time step")
plt.ylabel("Value")
plt.tight_layout()
plt.savefig("raw_data.png")
plt.show()


scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = scaler.fit_transform(X.reshape(-1, 1)).flatten()


def make_sequences(data, window):
    X_seq, y_seq = [], []
    for i in range(len(data) - window):
        X_seq.append(data[i:i+window])
        y_seq.append(data[i+window])
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)

WINDOW = 20  # tune this in part (b)
X_seq, y_seq = make_sequences(X_scaled, WINDOW)

# Train/val split (80/20)
split = int(len(X_seq) * 0.8)
X_train, X_val = X_seq[:split], X_seq[split:]
y_train, y_val = y_seq[:split], y_seq[split:]

print(f"Train: {X_train.shape}, Val: {X_val.shape}")

Select your choice of neural networks model that is suitable for this task and motivate it. Train your model to predict one step ahead data point, during training (see following Figure). Scale your data before training and scale them back to be able to compare your predictions with real measurements.

## TCN (Temporal Convolutional Network)

#### Motivation:

The dataset is a 1D physical signal of 1000 laser measurements with quasi-periodic spikes followed by sudden collapses, so each sample strongly depends on previous ones. We propose using a Temporal Convolutional Network (TCN). A TCN is a 1D causal convolutional network with dilated convolutions, which lets the receptive field grow exponentially with depth and allows the model to look far back in time without becoming as deep as a recurrent network. Compared to RNNs/GRU/LSTM, training a TCN is more stable because gradients flow through fixed-length convolutional paths instead of long recurrent unrolls, and the model parallelizes well over the sequence dimension. Residual connections inside each temporal block also help to keep the optimization stable on a small dataset like this one. For these reasons we believe a TCN is a good choice for predicting the next value in this laser sequence.

In [ ]:

# Hyperparameters
CHANNELS   = 32
NUM_BLOCKS = 4
KERNEL     = 3
DROPOUT    = 0.1
BATCH_SIZE = 32
EPOCHS     = 75
LR         = 1e-3
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# TCN building blocks
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x if self.chomp_size == 0 else x[:, :, :-self.chomp_size]

class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size, padding=pad, dilation=dilation),
            Chomp1d(pad), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad, dilation=dilation),
            Chomp1d(pad), nn.ReLU(), nn.Dropout(dropout),
        )
        self.down = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None
        self.act  = nn.ReLU()
    def forward(self, x):
        out = self.net(x)
        res = x if self.down is None else self.down(x)
        return self.act(out + res)

class TCNModel(nn.Module):
    def __init__(self, channels=32, num_blocks=4, kernel_size=3, dropout=0.1):
        super().__init__()
        layers = []
        in_ch = 1
        for i in range(num_blocks):
            layers.append(TemporalBlock(in_ch, channels, kernel_size, dilation=2**i, dropout=dropout))
            in_ch = channels
        self.tcn = nn.Sequential(*layers)
        self.fc  = nn.Linear(channels, 1)
    def forward(self, x):
        # x: (B, W, 1) -> (B, 1, W)
        h = self.tcn(x.transpose(1, 2))
        return self.fc(h[:, :, -1]).squeeze(-1)

# Prepare tensors
X_train_t = torch.tensor(X_train).unsqueeze(-1)
X_val_t   = torch.tensor(X_val).unsqueeze(-1)
y_train_t = torch.tensor(y_train)
y_val_t   = torch.tensor(y_val)

# DataLoaders
train_dl = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(TensorDataset(X_val_t,   y_val_t),   batch_size=64,         shuffle=False)

# Model, optimizer and loss function
model = TCNModel(channels=CHANNELS, num_blocks=NUM_BLOCKS, kernel_size=KERNEL, dropout=DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()

# Training loop
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):

    # Training phase
    model.train()
    batch_losses = []
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())
    train_losses.append(float(np.mean(batch_losses)))

    # Validation phase
    model.eval()
    with torch.no_grad():
        vb_losses = [criterion(model(xb.to(device)), yb.to(device)).item()
                     for xb, yb in val_dl]
    val_losses.append(float(np.mean(vb_losses)))

    if epoch % 5 == 0:
        print(f"Epoch {epoch:3d}/{EPOCHS} | Train MSE: {train_losses[-1]:.6f} | Val MSE: {val_losses[-1]:.6f}")

# Plot loss curves
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label="Train MSE")
ax.plot(val_losses,   label="Val MSE")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss (scaled space)")
ax.set_title("TCN Training History")
ax.legend()
plt.tight_layout()
plt.show()

# Generate predictions on full sequence and inverse scale back to original range
model.eval()
with torch.no_grad():
    X_all_t   = torch.tensor(X_seq).unsqueeze(-1).to(device)
    y_pred_sc = model(X_all_t).cpu().numpy()

y_pred = scaler.inverse_transform(y_pred_sc.reshape(-1, 1)).flatten()
y_true = scaler.inverse_transform(y_seq.reshape(-1, 1)).flatten()

# Align time indices — predictions start at WINDOW because
# the first window X[0:WINDOW] predicts X[WINDOW]
t_all = np.arange(WINDOW, len(X))
t_tr  = t_all[:split]
t_val = t_all[split:]

# Plot predictions vs real measurements
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(X, color="gray", alpha=0.45, linewidth=0.8, label="Real data (original scale)")
ax.plot(t_tr,  y_pred[:split], color="steelblue", linewidth=1.2, label="Train predictions")
ax.plot(t_val, y_pred[split:], color="orange",    linewidth=1.2, label="Val predictions")
ax.axvline(t_val[0], color="red", linestyle="--", linewidth=1.0, label="Train / Val split")
ax.set_title("TCN One-Step-Ahead Predictions vs Real Laser Measurements")
ax.set_xlabel("Time step")
ax.set_ylabel("Laser measurement value")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
"""
b: window size tuning
"""

# Reload Data
data = scipy.io.loadmat('../Xtrain.mat')
X = data['Xtrain'].flatten().astype(np.float32)
scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = scaler.fit_transform(X.reshape(-1, 1)).flatten()

def make_sequences(data, window):
    X_seq, y_seq = [], []
    for i in range(len(data) - window):
        X_seq.append(data[i:i+window])
        y_seq.append(data[i+window])
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# WINDOW SEARCH
WINDOWS = [5, 10, 20, 30, 50]
EPOCHS  = 75
LR      = 1e-3
results = {}

for W in WINDOWS:
    X_seq, y_seq = make_sequences(X_scaled, W)
    split = int(len(X_seq) * 0.8)
    X_tr, X_va = X_seq[:split], X_seq[split:]
    y_tr, y_va = y_seq[:split], y_seq[split:]

    tr_dl = DataLoader(TensorDataset(torch.tensor(X_tr).unsqueeze(-1),
                                     torch.tensor(y_tr)),
                       batch_size=32, shuffle=True)
    va_dl = DataLoader(TensorDataset(torch.tensor(X_va).unsqueeze(-1),
                                     torch.tensor(y_va)),
                       batch_size=64, shuffle=False)

    model = TCNModel(channels=CHANNELS, num_blocks=NUM_BLOCKS,
                     kernel_size=KERNEL, dropout=DROPOUT).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=LR)
    crit  = nn.MSELoss()

    best_val = float('inf')
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for xb, yb in tr_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_mse = np.mean([crit(model(xb.to(device)), yb.to(device)).item()
                               for xb, yb in va_dl])
        if val_mse < best_val:
            best_val = val_mse

    results[W] = best_val
    print(f"Window={W:3d}  best Val MSE={best_val:.6f}")

best_window = min(results, key=results.get)
print(f"\nBest window size: {best_window}  (Val MSE={results[best_window]:.6f})")

# bar chart
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([str(w) for w in WINDOWS], [results[w] for w in WINDOWS], color='steelblue')
ax.set_xlabel("Window size")
ax.set_ylabel("Best Val MSE (scaled)")
ax.set_title("Part (b) - Window size vs. Validation MSE")
plt.tight_layout()
plt.show()

In [ ]:
"""
c: recursive 200-step ahead forecast
"""
# Re-train final model on the full training series
W = best_window
X_seq, y_seq = make_sequences(X_scaled, W)

tr_dl = DataLoader(TensorDataset(torch.tensor(X_seq).unsqueeze(-1),
                                 torch.tensor(y_seq)),
                   batch_size=32, shuffle=True)

final_model = TCNModel(channels=CHANNELS, num_blocks=NUM_BLOCKS,
                       kernel_size=KERNEL, dropout=DROPOUT).to(device)
opt  = torch.optim.Adam(final_model.parameters(), lr=1e-3)
crit = nn.MSELoss()

for epoch in range(1, EPOCHS + 1):
    final_model.train()
    for xb, yb in tr_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        crit(final_model(xb), yb).backward()
        opt.step()

# Seed: the last W points of the full training series
seed = X_scaled[-W:].tolist()

final_model.eval()
recursive_preds_scaled = []

with torch.no_grad():
    window = seed.copy()
    for _ in range(200):
        inp = torch.tensor(window[-W:], dtype=torch.float32)
        inp = inp.unsqueeze(0).unsqueeze(-1).to(device)   # (1, W, 1)
        pred = final_model(inp).item()
        recursive_preds_scaled.append(pred)
        window.append(pred)

recursive_preds = scaler.inverse_transform(
    np.array(recursive_preds_scaled).reshape(-1, 1)).flatten()

#  plot
t_hist     = np.arange(len(X))
t_forecast = np.arange(len(X), len(X) + 200)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(t_hist, X, color='gray', alpha=0.5, lw=0.8, label='Historical data')
ax.axvline(len(X), color='red', linestyle='--', lw=1.0, label='Forecast start')
ax.plot(t_forecast, recursive_preds, color='darkorange', lw=1.5,
        label='200-step recursive forecast')
ax.set_title(f"Part (c) - Recursive 200-Step Forecast  (window={W})")
ax.set_xlabel("Time step")
ax.set_ylabel("Laser measurement value")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Forecast covers steps {t_forecast[0]} - {t_forecast[-1]}")
print(f"Predicted range: [{recursive_preds.min():.1f}, {recursive_preds.max():.1f}]")

In [ ]:
"""
d: evaluation on Xtest.mat (MAE, MSE, predicted vs real plot)
"""
test = scipy.io.loadmat('../Xtest.mat')
X_test = test['Xtest'].flatten().astype(np.float32)

mae = float(np.mean(np.abs(recursive_preds - X_test)))
mse = float(np.mean((recursive_preds - X_test) ** 2))
print(f"MAE: {mae:.4f}")
print(f"MSE: {mse:.4f}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(X_test,        color='black',     lw=1.2, label='Real test')
ax.plot(recursive_preds, color='darkorange', lw=1.2, label='TCN prediction')
ax.set_title(f"Part (d) - Predicted vs Real on Xtest.mat  (MAE={mae:.2f}, MSE={mse:.2f})")
ax.set_xlabel("Forecast step")
ax.set_ylabel("Laser measurement value")
ax.legend()
plt.tight_layout()
plt.show()